# CPU Benchmark Analysis — Data Cleaning

In this notebook we clean the dataset based on the issues identified during exploration.
No new analysis is done here — every decision made in this notebook is justified by
what we found in Notebook 1.

The issues we are addressing:
- powerPerf stored as string instead of float
- testDate stored as a year integer
- price, TDP and their derived columns have significant missing values
- category has multi-label values
- socket has 150+ inconsistently formatted unique values

The goal is to produce a clean, model-ready dataset by the end of this notebook.

In [1]:
# Mount Google Drive and load the dataset
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

path = '/content/drive/MyDrive/CPU Project/'
df = pd.read_csv(path + 'CPU_benchmark_v4.csv')

print("Dataset loaded successfully")
print(f"Shape: {df.shape}")

Mounted at /content/drive
Dataset loaded successfully
Shape: (3825, 12)


------

## Step 1 — Fix Data Types

The first thing we fix is data types. This is done before handling missing values
because some cleaning steps depend on the column being the correct type.

We have two issues identified in Notebook 1:
- powerPerf is stored as a string and needs to be converted to float
- testDate is a year stored as an integer which is already fine as is

In [2]:
# Convert powerPerf from string to float
# pd.to_numeric handles any edge cases safely with errors='coerce'
# coerce means any value that cannot be converted becomes NaN instead of crashing

df['powerPerf'] = pd.to_numeric(df['powerPerf'], errors='coerce')

# Confirm the fix
print("powerPerf dtype after conversion:", df['powerPerf'].dtype)
print("Sample values:", df['powerPerf'].dropna().head(5).tolist())

powerPerf dtype after conversion: float64
Sample values: [388.65, 315.49, 381.6, 299.9, 291.31]


-----

## Step 2 — Handle Missing Values

With data types fixed we now address missing values. The strategy for each column
depends on the nature of the column and how much data is missing.

Recall from Notebook 1:
- price is missing for 48.6% of rows
- cpuValue and threadValue are derived from price — missing for the same rows
- TDP is missing for 17.9% of rows
- powerPerf is derived from TDP — missing for the same rows

We handle each group separately.

### price, cpuValue and threadValue

Almost half the dataset has no price information. Simply filling in the missing
values with a mean or median would introduce too much noise given how skewed
the price distribution likely is across budget, mid-range, and high-end CPUs.

Since cpuValue and threadValue are calculated directly from price they do not
add independent information to the model. We will drop both derived columns
and keep price, filling its missing values with the median during the
feature engineering phase when we prepare the final model input.

In [3]:
# Drop cpuValue and threadValue — they are derived from price
# and do not add independent information to the model

df = df.drop(columns=['cpuValue', 'threadValue'])

print("Columns after dropping derived price columns:")
print(df.columns.tolist())

Columns after dropping derived price columns:
['cpuName', 'price', 'cpuMark', 'threadMark', 'TDP', 'powerPerf', 'cores', 'testDate', 'socket', 'category']


----

### TDP and powerPerf

TDP is missing for 17.9% of rows. Unlike price, this is a manageable amount.
Dropping these rows would lose 685 CPUs which is significant but not catastrophic.
However since powerPerf is calculated directly from TDP it does not add independent
information — it is just cpuMark divided by TDP. We drop powerPerf and keep TDP,
filling its missing values with the median.

In [4]:
# Drop powerPerf — derived from TDP and adds no independent information
df = df.drop(columns=['powerPerf'])

print("Columns after dropping powerPerf:")
print(df.columns.tolist())

Columns after dropping powerPerf:
['cpuName', 'price', 'cpuMark', 'threadMark', 'TDP', 'cores', 'testDate', 'socket', 'category']


In [5]:
# Fill missing TDP values with the median
# Median is preferred over mean here because of the skewed distribution we saw earlier

tdp_median = df['TDP'].median()
df['TDP'] = df['TDP'].fillna(tdp_median)

print(f"TDP median used for imputation: {tdp_median}")
print(f"TDP missing values remaining: {df['TDP'].isnull().sum()}")

TDP median used for imputation: 53.0
TDP missing values remaining: 0


------

### price

Price is missing for 48.6% of rows. We keep the column because price is a
meaningful feature — it reflects market positioning of a CPU which correlates
with performance. Missing values will be filled with the median price, grouped
by category. This is a better estimate than a single global median because
laptop and server CPUs tend to have very different price ranges from desktop CPUs.

In [6]:
# Fill missing price values with the median price within each category group
# This gives a more accurate estimate than a single global median

df['price'] = df.groupby('category')['price'].transform(
    lambda x: x.fillna(x.median())
)

# Some categories may still have all nulls — fill those with the global median
df['price'] = df['price'].fillna(df['price'].median())

print(f"Price missing values remaining: {df['price'].isnull().sum()}")
print(f"Global median price used as fallback: {df['price'].median()}")

Price missing values remaining: 0
Global median price used as fallback: 159.99


-----

## Step 3 — Clean the Category Column

Recall from Notebook 1 that category contains multi-label values such as
"Desktop, Server" and "Laptop, Mobile/Embedded". A machine learning model
cannot work with these directly.

Our approach is to simplify each value down to a single primary category.
We assign each CPU to one of four clean labels — Desktop, Laptop, Server,
or Other — based on what appears in its category string.

In [7]:
# Simplify category into a single primary label
# We check each value in order of priority — Server first since server CPUs
# are a distinct and important group, then Desktop, then Laptop, then Other

def simplify_category(value):
    if 'Server' in str(value):
        return 'Server'
    elif 'Desktop' in str(value):
        return 'Desktop'
    elif 'Laptop' in str(value):
        return 'Laptop'
    else:
        return 'Other'

df['category'] = df['category'].apply(simplify_category)

print("Unique categories after cleaning:")
print(df['category'].value_counts())

Unique categories after cleaning:
category
Laptop     1176
Desktop    1159
Server      805
Other       685
Name: count, dtype: int64


### Category Distribution After Cleaning

The dataset is reasonably balanced across the four categories. Laptop and Desktop
CPUs make up the largest groups with roughly 1,100 each, followed by Server with
805 and Other with 685. The Other category captures Mobile/Embedded and Unknown
CPUs that do not fit cleanly into the main three groups.

----

## Step 4 — Clean the Socket Column

The socket column has over 150 unique values, many of which represent the same
socket written in different formats. For example LGA1700, FCLGA1700, and BGA1700
all refer to the same physical socket.

Standardizing every single socket name would be extremely tedious and the socket
column itself may not be a strong predictor of cpuMark on its own. Instead we
take a practical approach — we group rare sockets that appear very infrequently
into an "Other" category and standardize only the most common ones.

This reduces noise without losing meaningful information.

In [8]:
# First let us see how many sockets appear fewer than 10 times
# These rare sockets add noise without adding useful signal to the model

socket_counts = df['socket'].value_counts()

print(f"Total unique socket values: {len(socket_counts)}")
print(f"Sockets appearing fewer than 10 times: {(socket_counts < 10).sum()}")
print(f"Sockets appearing 10 or more times: {(socket_counts >= 10).sum()}")

Total unique socket values: 203
Sockets appearing fewer than 10 times: 126
Sockets appearing 10 or more times: 77


### Socket Cleaning Strategy

126 out of 203 unique socket values appear fewer than 10 times in the dataset.
Keeping all of them would add significant noise to the model. Our approach is
to group any socket that appears fewer than 10 times into an "Other" category
and keep only the 77 sockets that appear frequently enough to be meaningful.

In [9]:
# Replace rare sockets with 'Other'
# Any socket appearing fewer than 10 times is not frequent enough to be meaningful

socket_counts = df['socket'].value_counts()
rare_sockets = socket_counts[socket_counts < 10].index

df['socket'] = df['socket'].apply(lambda x: 'Other' if x in rare_sockets else x)

print(f"Unique socket values after cleaning: {df['socket'].nunique()}")
print()
print("Top 10 most common sockets:")
print(df['socket'].value_counts().head(10))

Unique socket values after cleaning: 78

Top 10 most common sockets:
socket
unknown        788
Other          380
AM4            133
LGA1155        103
FCLGA1200       97
LGA775          89
FCLGA1151-2     86
FCLGA3647       84
S1              83
LGA1150         82
Name: count, dtype: int64


In [10]:
# Replace 'unknown' with 'Other' since it carries no real information
df['socket'] = df['socket'].replace('unknown', 'Other')

# Standardize common socket names that appear in multiple formats
socket_mapping = {
    'FCLGA1200'  : 'LGA1200',
    'FCLGA1700'  : 'LGA1700',
    'BGA1700'    : 'LGA1700',
    'FCLGA1151-2': 'LGA1151',
    'FCLGA1151'  : 'LGA1151',
    'LGA 1700'   : 'LGA1700',
    'LGA 3647'   : 'LGA3647',
    'FCLGA3647'  : 'LGA3647',
    'FCLGA-3647' : 'LGA3647',
    'FCLGA2066'  : 'LGA2066',
    'FCLGA-2066' : 'LGA2066',
    'LGA 2066'   : 'LGA2066',
    'FCLGA2011-3': 'LGA2011',
    'FCLGA2011'  : 'LGA2011',
    'LGA2011-v3' : 'LGA2011',
    'LGA 2011'   : 'LGA2011',
    'FCLGA1150'  : 'LGA1150',
    'FCLGA1156'  : 'LGA1156',
    'FCLGA1155'  : 'LGA1155',
    'sTR4'       : 'TR4',
}

df['socket'] = df['socket'].replace(socket_mapping)

print(f"Unique socket values after standardization: {df['socket'].nunique()}")
print()
print("Top 10 most common sockets:")
print(df['socket'].value_counts().head(10))

Unique socket values after standardization: 70

Top 10 most common sockets:
socket
Other      1168
LGA2011     184
LGA1151     144
AM4         133
LGA1155     119
LGA1150     115
LGA1200      97
LGA775       89
LGA3647      84
S1           83
Name: count, dtype: int64


### Socket Column After Cleaning

After replacing unknown values and standardizing inconsistent formatting, the socket
column is down to 70 unique values. The most common sockets now appear under a single
consistent name — for example FCLGA1700, BGA1700, and LGA 1700 are all consolidated
into LGA1700.

The Other category now holds 1,168 CPUs — a combination of the original rare sockets
and the unknown entries. While this is a large Other group, it is preferable to keeping
hundreds of near-unique socket values that would add noise without signal.

----

## Step 5 — Final Check

With all cleaning steps complete we do a final check to confirm the dataset is
in good shape before saving it.

In [11]:
# Final check — confirm shape, data types, and remaining missing values

print("Shape:", df.shape)
print()
print("Data types:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())

Shape: (3825, 9)

Data types:
cpuName        object
price         float64
cpuMark         int64
threadMark      int64
TDP           float64
cores           int64
testDate        int64
socket         object
category       object
dtype: object

Missing values:
cpuName       0
price         0
cpuMark       0
threadMark    0
TDP           0
cores         0
testDate      0
socket        0
category      0
dtype: int64


### Final Check Results

The cleaned dataset has 3,825 rows and 9 columns with zero missing values across
all columns. Data types are correct — numeric columns are stored as int64 or float64
and categorical columns are stored as object.

The dataset is now ready to be saved and passed to the next notebook for
feature engineering and modeling.

-----

## Save the Cleaned Dataset

In [12]:
# Save the cleaned dataset to Google Drive
# We save it as a new file to preserve the original raw data

df.to_csv(path + 'CPU_benchmark_cleaned.csv', index=False)

print("Cleaned dataset saved successfully")
print(f"File: CPU_benchmark_cleaned.csv")
print(f"Shape: {df.shape}")

Cleaned dataset saved successfully
File: CPU_benchmark_cleaned.csv
Shape: (3825, 9)


## Summary

This notebook addressed all data quality issues identified during exploration.

- powerPerf and the derived columns cpuValue and threadValue were dropped
  since they carry no independent information beyond what price and TDP already provide
- Missing TDP values were filled with the median
- Missing price values were filled with the median within each category group
- category was simplified from multi-label strings into four clean labels
- socket was standardized from 203 inconsistent values down to 70 clean ones
- All unknown socket values were grouped into Other

The cleaned file CPU_benchmark_cleaned.csv is saved to the project folder
and will be used as the starting point for Notebook 3.